# GEO Cleaning - Expression và Clinical Metadata

Mục tiêu: chuẩn hóa dữ liệu GEO từ HDFS raw sang HDFS refined ở dạng Parquet.  
Input: `/drugtarget/data/raw/geo/expression` và `/drugtarget/data/raw/geo/metadata`.  
Output: `/drugtarget/data/refined/geo/...`.  
Version: `v1`


In [ ]:
from datetime import datetime
import glob
import json
import os
import socket
import sys
import time
from urllib import request as urlrequest

# Kernel Jupyter có thể chưa có pyspark trong PYTHONPATH.
# Vì Spark đã được cài local, cell này tự thêm thư viện Spark/Py4J trước khi import PySpark.
SPARK_HOME = os.environ.get("SPARK_HOME", "/home/dis/spark-3.5.1-bin-hadoop3")
os.environ.setdefault("SPARK_HOME", SPARK_HOME)
spark_python_dir = os.path.join(SPARK_HOME, "python")
if os.path.isdir(spark_python_dir) and spark_python_dir not in sys.path:
    sys.path.insert(0, spark_python_dir)
for py4j_zip in glob.glob(os.path.join(SPARK_HOME, "python", "lib", "py4j-*.zip")):
    if py4j_zip not in sys.path:
        sys.path.insert(0, py4j_zip)

from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F, types as T


# Cấu hình Spark cluster hiện tại. Notebook ưu tiên chạy song song trên worker,
# nhưng sẽ tự fallback về local nếu cluster không có worker/executor khả dụng.
SPARK_MASTER_HOST = "master11"
SPARK_MASTER_PORT = 7077
SPARK_CLUSTER_MASTER = f"spark://{SPARK_MASTER_HOST}:{SPARK_MASTER_PORT}"
SPARK_MASTER_WEB_UI = f"http://{SPARK_MASTER_HOST}:8080"
SPARK_DRIVER_HOST = "master11"

# Profile an toàn vì dữ liệu GEO expression rộng và worker trong môi trường này nhỏ.
# Các giá trị này giữ số task đồng thời vừa phải để tránh nghẽn CPU/RAM/I/O HDFS.
SAFE_DRIVER_MEMORY = "3g"
SAFE_EXECUTOR_MEMORY = "2g"
SAFE_EXECUTOR_CORES = 1
SAFE_MAX_CORES = 2
SAFE_DEFAULT_PARALLELISM = 32
SAFE_SHUFFLE_PARTITIONS = 32
SAFE_MAX_PARTITION_BYTES = "64m"
SAFE_EXPRESSION_OUTPUT_PARTITIONS = 8


def _is_spark_master_reachable(host: str, port: int, timeout_seconds: int = 2) -> bool:
    """Kiểm tra Spark master có mở cổng nhận application hay không."""
    try:
        with socket.create_connection((host, port), timeout=timeout_seconds):
            return True
    except OSError:
        return False


def _get_spark_master_status(timeout_seconds: int = 2) -> dict:
    """Đọc JSON từ Spark Master UI để biết cluster có worker sống hay không."""
    try:
        with urlrequest.urlopen(f"{SPARK_MASTER_WEB_UI}/json/", timeout=timeout_seconds) as response:
            return json.loads(response.read().decode("utf-8"))
    except Exception as exc:
        print(f"Không đọc được Spark Master UI JSON, sẽ cân nhắc fallback local: {exc}")
        return {}


def _get_alive_worker_count(master_status: dict) -> int:
    """Đếm worker đang ALIVE; nếu field state không có thì coi worker là khả dụng."""
    workers = master_status.get("workers") or []
    alive_workers = []
    for worker in workers:
        worker_state = str(worker.get("state", "ALIVE")).upper()
        worker_cores = int(worker.get("cores", worker.get("coresfree", 0)) or 0)
        if worker_state == "ALIVE" and worker_cores > 0:
            alive_workers.append(worker)
    return len(alive_workers)


def _stop_existing_spark_session() -> None:
    """Dừng SparkSession cũ trong kernel để tránh reuse sai master khi chạy lại notebook."""
    existing_spark = globals().get("spark")
    if existing_spark is None:
        return
    try:
        existing_spark.stop()
        time.sleep(2)
    except Exception as exc:
        print(f"Không thể stop SparkSession cũ, tiếp tục tạo session mới: {exc}")


def _get_executor_count(spark_session):
    """Đếm executor của app hiện tại, không tính driver."""
    try:
        memory_status_size = spark_session.sparkContext._jsc.sc().getExecutorMemoryStatus().size()
        return max(memory_status_size - 1, 0)
    except Exception:
        return "unknown"


def _wait_for_executors(spark_session, timeout_seconds: int = 20):
    """Đợi worker cấp executor trước khi kết luận app cluster có chạy song song được hay không."""
    deadline = time.time() + timeout_seconds
    executor_count = _get_executor_count(spark_session)
    while isinstance(executor_count, int) and executor_count == 0 and time.time() < deadline:
        time.sleep(1)
        executor_count = _get_executor_count(spark_session)
    return executor_count


def _build_spark_session(app_name: str, master_url: str) -> SparkSession:
    """Tạo SparkSession với cấu hình giảm tải, dùng chung cho cluster và local fallback."""
    return (
        SparkSession.builder
        .appName(app_name)
        .master(master_url)
        .config("spark.driver.host", SPARK_DRIVER_HOST)
        .config("spark.driver.memory", SAFE_DRIVER_MEMORY)
        .config("spark.executor.memory", SAFE_EXECUTOR_MEMORY)
        .config("spark.executor.cores", str(SAFE_EXECUTOR_CORES))
        .config("spark.cores.max", str(SAFE_MAX_CORES))
        .config("spark.default.parallelism", str(SAFE_DEFAULT_PARALLELISM))
        .config("spark.sql.shuffle.partitions", str(SAFE_SHUFFLE_PARTITIONS))
        .config("spark.sql.files.maxPartitionBytes", SAFE_MAX_PARTITION_BYTES)
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .config("spark.pyspark.python", "python3")
        .config("spark.executorEnv.PYSPARK_PYTHON", "python3")
        .getOrCreate()
    )


def create_spark_session(app_name: str):
    """Ưu tiên cluster nếu có worker/executor; nếu không thì WARN và chạy local[*]."""
    _stop_existing_spark_session()

    master_reachable = _is_spark_master_reachable(SPARK_MASTER_HOST, SPARK_MASTER_PORT)
    master_status = _get_spark_master_status() if master_reachable else {}
    alive_worker_count = _get_alive_worker_count(master_status) if master_reachable else 0

    if not master_reachable or alive_worker_count == 0:
        print("WARN: Không tìm thấy Spark worker khả dụng; chạy bằng local[*].")
        spark_session = _build_spark_session(app_name, "local[*]")
        spark_mode = "LOCAL_FALLBACK"
        executor_count = 0
    else:
        spark_session = _build_spark_session(app_name, SPARK_CLUSTER_MASTER)
        spark_mode = "CLUSTER"
        executor_count = _wait_for_executors(spark_session)

        # Nếu master có worker nhưng app không nhận được executor, dừng app cluster và chuyển local.
        if isinstance(executor_count, int) and executor_count == 0:
            print("WARN: Không tìm thấy Spark worker khả dụng; chạy bằng local[*].")
            spark_session.stop()
            time.sleep(2)
            spark_session = _build_spark_session(app_name, "local[*]")
            spark_mode = "LOCAL_FALLBACK"
            executor_count = 0

    # In thông tin vận hành để người chạy notebook biết đang dùng cluster hay local.
    sc = spark_session.sparkContext
    print(f"Spark mode: {spark_mode}")
    print(f"Spark master: {sc.master}")
    print(f"Spark master UI: {SPARK_MASTER_WEB_UI}")
    print(f"Spark app name: {sc.appName}")
    print(f"Spark app id: {sc.applicationId}")
    print(f"Spark UI URL: {sc.uiWebUrl}")
    print(f"Spark executors: {executor_count}")
    print(
        "Safe Spark profile: "
        f"max_cores={SAFE_MAX_CORES}, executor_cores={SAFE_EXECUTOR_CORES}, "
        f"shuffle_partitions={SAFE_SHUFFLE_PARTITIONS}, "
        f"expression_output_partitions={SAFE_EXPRESSION_OUTPUT_PARTITIONS}"
    )

    return spark_session, spark_mode, executor_count


# Khởi tạo SparkSession cho job chuẩn hóa GEO.
spark, SPARK_MODE, SPARK_EXECUTOR_COUNT = create_spark_session("geo_cleaning_refined_outputs_v1")

# Khai báo HDFS root và các path input/output. Các output đều nằm dưới refined/geo.
HDFS_BASE_URI = "hdfs://master11:9000"
RAW_GEO_ROOT = f"{HDFS_BASE_URI}/drugtarget/data/raw/geo"
REFINED_GEO_ROOT = f"{HDFS_BASE_URI}/drugtarget/data/refined/geo"
VERSION = "v1"

EXPRESSION_INPUT_PATH = f"{RAW_GEO_ROOT}/expression/geo_ex.txt"
METADATA_INPUT_DIR = f"{RAW_GEO_ROOT}/metadata"
METADATA_EXPECTED_BASENAME = "acc.cgi?acc=GSE67639&targ=self&form=text&view=full"

OUTPUT_EXPRESSION = f"{REFINED_GEO_ROOT}/expression/version={VERSION}"
OUTPUT_METADATA = f"{REFINED_GEO_ROOT}/metadata/version={VERSION}"
OUTPUT_QC = f"{REFINED_GEO_ROOT}/qc/version={VERSION}"
GEO_CHECKPOINT_DIR = f"{HDFS_BASE_URI}/drugtarget/tmp/checkpoints/geo_cleaning"

# Ép Hadoop/Spark dùng đúng NameNode HDFS, tránh hiểu path /drugtarget là local filesystem.
spark.sparkContext._jsc.hadoopConfiguration().set("fs.defaultFS", HDFS_BASE_URI)
spark.sparkContext.setCheckpointDir(GEO_CHECKPOINT_DIR)

print(f"RAW_GEO_ROOT         : {RAW_GEO_ROOT}")
print(f"REFINED_GEO_ROOT     : {REFINED_GEO_ROOT}")
print(f"EXPRESSION_INPUT_PATH: {EXPRESSION_INPUT_PATH}")
print(f"METADATA_INPUT_DIR   : {METADATA_INPUT_DIR}")
print(f"OUTPUT_EXPRESSION    : {OUTPUT_EXPRESSION}")
print(f"OUTPUT_METADATA      : {OUTPUT_METADATA}")
print(f"OUTPUT_QC            : {OUTPUT_QC}")


## 1. Hàm tiện ích và kiểm tra path HDFS


In [ ]:
# Lấy FileSystem của Hadoop để kiểm tra input/output trực tiếp trên HDFS.
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
Path = spark._jvm.org.apache.hadoop.fs.Path


def hdfs_exists(path: str) -> bool:
    """Kiểm tra path HDFS có tồn tại hay không."""
    return fs.exists(Path(path))


def validate_output_path(path: str) -> None:
    """Chặn mọi output không nằm dưới refined/geo để tránh ghi nhầm raw hoặc layer khác."""
    allowed_prefix = REFINED_GEO_ROOT.rstrip("/") + "/"
    if not path.startswith(allowed_prefix):
        raise ValueError(
            "Output path không hợp lệ. Chỉ được ghi dưới "
            + allowed_prefix
            + f", nhưng nhận được: {path}"
        )


def resolve_metadata_input_path() -> str:
    """Tìm file metadata thật trên HDFS, xử lý cả trường hợp tên file có khoảng trắng cuối."""
    metadata_dir = Path(METADATA_INPUT_DIR)
    if not fs.exists(metadata_dir):
        raise FileNotFoundError(f"Thiếu thư mục metadata HDFS: {METADATA_INPUT_DIR}")

    for status in fs.listStatus(metadata_dir):
        if not status.isFile():
            continue
        file_name = status.getPath().getName()
        if file_name.rstrip() == METADATA_EXPECTED_BASENAME:
            return status.getPath().toString()

    raise FileNotFoundError(
        "Không tìm thấy file metadata GEO có basename: "
        + METADATA_EXPECTED_BASENAME
        + f" trong {METADATA_INPUT_DIR}"
    )


def assert_required_input_paths() -> None:
    """Báo lỗi sớm nếu thiếu expression hoặc metadata trước khi chạy job tốn tài nguyên."""
    if not hdfs_exists(EXPRESSION_INPUT_PATH):
        raise FileNotFoundError(f"Thiếu file expression HDFS: {EXPRESSION_INPUT_PATH}")

    resolved_metadata_path = resolve_metadata_input_path()
    if not hdfs_exists(resolved_metadata_path):
        raise FileNotFoundError(f"Thiếu file metadata HDFS: {resolved_metadata_path}")


def make_qc_df(metric_rows):
    """Tạo DataFrame QC nhỏ với schema ổn định để ghi Parquet."""
    schema = T.StructType([
        T.StructField("metric_name", T.StringType(), False),
        T.StructField("metric_value", T.StringType(), False),
        T.StructField("details", T.StringType(), True),
        T.StructField("created_at", T.StringType(), False),
    ])
    created_at = datetime.utcnow().isoformat(timespec="seconds") + "Z"
    normalized_rows = [
        (str(metric_name), str(metric_value), None if details is None else str(details), created_at)
        for metric_name, metric_value, details in metric_rows
    ]
    return spark.createDataFrame(normalized_rows, schema=schema)


# Validate input và output path ngay từ đầu để fail fast nếu sai cấu hình.
assert_required_input_paths()
METADATA_INPUT_PATH = resolve_metadata_input_path()
for output_path in [OUTPUT_EXPRESSION, OUTPUT_METADATA, OUTPUT_QC]:
    validate_output_path(output_path)

print("Đã validate input và output path thành công")
print(f"METADATA_INPUT_PATH  : {METADATA_INPUT_PATH}")


## 2. Chuẩn hóa bảng expression


In [ ]:
# Đọc expression bằng RDD text để tự bỏ chính xác các dòng mô tả đầu file.
# File này có vài dòng đầu không phải bảng dữ liệu; header thật bắt đầu bằng ID_REF<TAB>Description<TAB>.
expression_lines_with_index = (
    spark.sparkContext
    .textFile(EXPRESSION_INPUT_PATH, minPartitions=SAFE_DEFAULT_PARALLELISM)
    .zipWithIndex()
    .persist(StorageLevel.DISK_ONLY)
)

# Tìm vị trí header thật của bảng expression. Dùng startswith để không phụ thuộc số lượng sample phía sau.
expression_header_candidates = (
    expression_lines_with_index
    .filter(lambda row: row[0].startswith("ID_REF	Description	"))
    .map(lambda row: row[1])
    .collect()
)

if not expression_header_candidates:
    raise ValueError("Không tìm thấy header expression bắt đầu bằng 'ID_REF\tDescription\t'")

expression_header_index = min(expression_header_candidates)

# Giữ từ header thật trở xuống, đồng thời bỏ ký tự CR nếu file có line ending kiểu Windows.
expression_table_lines = (
    expression_lines_with_index
    .filter(lambda row: row[1] >= expression_header_index)
    .map(lambda row: row[0].rstrip("\r"))
)

# Đọc lại phần bảng TSV bằng Spark CSV reader để xử lý delimiter tab và header đúng chuẩn.
expression_raw = (
    spark.read
    .option("sep", "	")
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .csv(expression_table_lines)
    .persist(StorageLevel.DISK_ONLY)
)

# Validate schema nguồn trước khi clean để tránh silent data loss nếu file đổi format.
required_expression_columns = {"ID_REF", "Description"}
missing_expression_columns = required_expression_columns - set(expression_raw.columns)
if missing_expression_columns:
    raise ValueError("Expression thiếu cột bắt buộc: " + ", ".join(sorted(missing_expression_columns)))

# Clean đúng yêu cầu: bỏ Description và đổi ID_REF thành gene_name.
# Không đổi tên các cột sample, không lower/snake_case, không chuẩn hóa thêm giá trị expression.
geo_expression_refined = (
    expression_raw
    .drop("Description")
    .withColumnRenamed("ID_REF", "gene_name")
    .persist(StorageLevel.DISK_ONLY)
)

# Kiểm tra lại schema sau clean.
if "gene_name" not in geo_expression_refined.columns:
    raise ValueError("Expression sau clean thiếu cột gene_name")
if "ID_REF" in geo_expression_refined.columns or "Description" in geo_expression_refined.columns:
    raise ValueError("Expression sau clean vẫn còn ID_REF hoặc Description")

expression_metrics = geo_expression_refined.agg(F.count("*").alias("row_count")).collect()[0]
expression_row_count = expression_metrics["row_count"]
expression_column_count = len(geo_expression_refined.columns)

if expression_row_count == 0 or expression_column_count == 0:
    raise ValueError("Expression sau clean không có dòng hoặc không có cột")

print(f"Expression header index bị bỏ qua trước data: {expression_header_index}")
print(f"Expression rows   : {expression_row_count}")
print(f"Expression columns: {expression_column_count}")
print("Preview expression refined:")
geo_expression_refined.show(5, truncate=False)


## 3. Chuẩn hóa bảng metadata bệnh nhân


In [ ]:
# File metadata là GEO SOFT: phía trên có mô tả series, còn bảng bệnh nhân nằm giữa marker begin/end.
# Bước này chỉ lấy phần bảng bệnh nhân; không chuẩn hóa tên cột và không đổi chữ hoa/thường.
METADATA_TABLE_BEGIN_MARKER = "!series_table_begin = Clinical_annotations_curated"
METADATA_TABLE_END_MARKER = "!series_table_end"

metadata_lines_with_index = (
    spark.sparkContext
    .textFile(METADATA_INPUT_PATH, minPartitions=1)
    .zipWithIndex()
    .persist(StorageLevel.DISK_ONLY)
)

metadata_begin_candidates = (
    metadata_lines_with_index
    .filter(lambda row: row[0].strip() == METADATA_TABLE_BEGIN_MARKER)
    .map(lambda row: row[1])
    .collect()
)
metadata_end_candidates = (
    metadata_lines_with_index
    .filter(lambda row: row[0].strip() == METADATA_TABLE_END_MARKER)
    .map(lambda row: row[1])
    .collect()
)

if not metadata_begin_candidates:
    raise ValueError(f"Không tìm thấy marker metadata begin: {METADATA_TABLE_BEGIN_MARKER}")
if not metadata_end_candidates:
    raise ValueError(f"Không tìm thấy marker metadata end: {METADATA_TABLE_END_MARKER}")

metadata_begin_index = min(metadata_begin_candidates)
metadata_end_index = min(index for index in metadata_end_candidates if index > metadata_begin_index)

# Lấy các dòng nằm giữa begin và end: dòng đầu tiên trong đoạn này là header thật của bảng bệnh nhân.
metadata_table_lines = (
    metadata_lines_with_index
    .filter(lambda row: metadata_begin_index < row[1] < metadata_end_index)
    .map(lambda row: row[0].rstrip("\r"))
)

metadata_raw = (
    spark.read
    .option("sep", "	")
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .csv(metadata_table_lines)
    .persist(StorageLevel.DISK_ONLY)
)

# Clean đúng yêu cầu: chỉ bỏ Original Study và Cohort.
# Giữ nguyên tên cột gốc của GEO, bao gồm MergeGroup và các cột có chữ hoa/thường ban đầu.
metadata_columns_to_drop = ["Original Study", "Cohort"]
missing_metadata_drop_columns = [column for column in metadata_columns_to_drop if column not in metadata_raw.columns]
if missing_metadata_drop_columns:
    raise ValueError("Metadata thiếu cột cần bỏ: " + ", ".join(missing_metadata_drop_columns))

geo_metadata_refined = (
    metadata_raw
    .drop(*metadata_columns_to_drop)
    .persist(StorageLevel.DISK_ONLY)
)

# Validate metadata sau clean: chỉ mất đúng hai cột yêu cầu, MergeGroup vẫn còn nguyên tên.
for dropped_column in metadata_columns_to_drop:
    if dropped_column in geo_metadata_refined.columns:
        raise ValueError(f"Metadata sau clean vẫn còn cột cần bỏ: {dropped_column}")
if "MergeGroup" not in geo_metadata_refined.columns:
    raise ValueError("Metadata sau clean không còn cột MergeGroup, trái yêu cầu giữ nguyên cột còn lại")

metadata_row_count = geo_metadata_refined.count()
metadata_column_count = len(geo_metadata_refined.columns)

if metadata_row_count == 0 or metadata_column_count == 0:
    raise ValueError("Metadata sau clean không có dòng hoặc không có cột")

print(f"Metadata table begin index: {metadata_begin_index}")
print(f"Metadata table end index  : {metadata_end_index}")
print(f"Metadata rows             : {metadata_row_count}")
print(f"Metadata columns          : {metadata_column_count}")
print("Preview metadata refined:")
geo_metadata_refined.show(5, truncate=False)


## 4. Tạo QC và ghi Parquet vào refined/geo


In [ ]:
# Tạo bảng QC để ghi lại số dòng/cột, Spark mode và các cột đã bỏ.
qc_rows = [
    ("spark_mode", SPARK_MODE, "CLUSTER nếu có worker/executor, LOCAL_FALLBACK nếu phải chạy local"),
    ("spark_executor_count", SPARK_EXECUTOR_COUNT, "Số executor của app Spark hiện tại, không tính driver"),
    ("expression_input_path", EXPRESSION_INPUT_PATH, "Đường dẫn expression raw trên HDFS"),
    ("metadata_input_path", METADATA_INPUT_PATH, "Đường dẫn metadata raw thực tế trên HDFS"),
    ("expression_header_index", expression_header_index, "Số dòng đầu file expression bị bỏ trước header thật"),
    ("expression_row_count", expression_row_count, "Số dòng expression sau clean"),
    ("expression_column_count", expression_column_count, "Số cột expression sau clean"),
    ("expression_dropped_columns", "Description", "Cột bị bỏ khỏi expression"),
    ("expression_renamed_columns", "ID_REF -> gene_name", "Cột expression được đổi tên"),
    ("metadata_begin_index", metadata_begin_index, "Dòng marker bắt đầu bảng metadata bệnh nhân"),
    ("metadata_end_index", metadata_end_index, "Dòng marker kết thúc bảng metadata bệnh nhân"),
    ("metadata_row_count", metadata_row_count, "Số dòng metadata sau clean"),
    ("metadata_column_count", metadata_column_count, "Số cột metadata sau clean"),
    ("metadata_dropped_columns", ", ".join(metadata_columns_to_drop), "Các cột bị bỏ khỏi metadata"),
]
geo_cleaning_qc = make_qc_df(qc_rows)

print("QC preview:")
geo_cleaning_qc.show(truncate=False)

# Ghi expression Parquet với số partition vừa phải để không tạo quá nhiều task/file nhỏ.
# Metadata và QC nhỏ nên coalesce(1) để dễ kiểm tra bằng HDFS CLI hoặc Spark sau này.
(
    geo_expression_refined
    .coalesce(SAFE_EXPRESSION_OUTPUT_PARTITIONS)
    .write
    .mode("overwrite")
    .parquet(OUTPUT_EXPRESSION)
)

(
    geo_metadata_refined
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(OUTPUT_METADATA)
)

(
    geo_cleaning_qc
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(OUTPUT_QC)
)

print(f"Đã ghi expression Parquet: {OUTPUT_EXPRESSION}")
print(f"Đã ghi metadata Parquet  : {OUTPUT_METADATA}")
print(f"Đã ghi QC Parquet        : {OUTPUT_QC}")


## 5. Đọc lại Parquet để kiểm tra output


In [ ]:
# Đọc lại output từ HDFS để xác nhận Parquet đã ghi được và schema đúng với yêu cầu.
for output_path in [OUTPUT_EXPRESSION, OUTPUT_METADATA, OUTPUT_QC]:
    validate_output_path(output_path)
    if not hdfs_exists(output_path):
        raise FileNotFoundError(f"Chưa thấy output sau khi ghi: {output_path}")

expression_check = spark.read.parquet(OUTPUT_EXPRESSION)
metadata_check = spark.read.parquet(OUTPUT_METADATA)
qc_check = spark.read.parquet(OUTPUT_QC)

expression_check_count = expression_check.count()
metadata_check_count = metadata_check.count()
qc_check_count = qc_check.count()

if expression_check_count == 0:
    raise ValueError("Expression Parquet đọc lại có 0 dòng")
if metadata_check_count == 0:
    raise ValueError("Metadata Parquet đọc lại có 0 dòng")
if qc_check_count == 0:
    raise ValueError("QC Parquet đọc lại có 0 dòng")

if "gene_name" not in expression_check.columns:
    raise ValueError("Expression Parquet thiếu gene_name")
if "ID_REF" in expression_check.columns or "Description" in expression_check.columns:
    raise ValueError("Expression Parquet vẫn còn ID_REF hoặc Description")
if "Original Study" in metadata_check.columns or "Cohort" in metadata_check.columns:
    raise ValueError("Metadata Parquet vẫn còn Original Study hoặc Cohort")
if "MergeGroup" not in metadata_check.columns:
    raise ValueError("Metadata Parquet thiếu MergeGroup")

print("Kiểm tra output Parquet thành công")
print(f"Expression Parquet rows: {expression_check_count}, columns: {len(expression_check.columns)}")
print(f"Metadata Parquet rows  : {metadata_check_count}, columns: {len(metadata_check.columns)}")
print(f"QC Parquet rows        : {qc_check_count}")

print("Expression schema:")
expression_check.printSchema()
print("Metadata schema:")
metadata_check.printSchema()


In [ ]:
# Dừng Spark app sau khi hoàn tất để trả tài nguyên cho worker/local runtime.
try:
    spark.stop()
    print("Đã stop Spark app GEO")
except Exception as exc:
    print(f"Không thể stop Spark app GEO: {exc}")
